# 04 - IV First Stage

Validate the smoke instrument. First stage: does smoke exposure predict PM2.5? Report F-statistic — rule of thumb is F > 10 for a strong instrument.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

In [ ]:
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

In [ ]:
panel_iv = panel.dropna(subset=["leaid","year","pm25_annual_mean","smoke_days","test_score_mean"]).copy()
panel_iv["year"] = panel_iv["year"].astype(int)

print(f"IV sample: {len(panel_iv):,} district-years")
print(f"           {panel_iv['leaid'].nunique():,} districts")

## First stage regression: PM2.5 ~ smoke_days + FE

In [ ]:
idx = panel_iv.set_index(["leaid","year"])

fs = PanelOLS(
    idx["pm25_annual_mean"],
    idx[["smoke_days"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

b_fs = fs.params["smoke_days"]
f_stat = fs.f_statistic.stat if hasattr(fs, "f_statistic") else None

print(f"First stage: β(smoke_days) = {b_fs:.4f}")
print(f"F-statistic: {f_stat}")
print()
if f_stat and f_stat > 10:
    print("✓ F > 10: instrument is relevant (strong first stage)")
elif f_stat:
    print("⚠️  F < 10: weak instrument — interpret IV estimates with caution")
print()
print(fs.summary.tables[1])

## First stage visualization

In [ ]:
# Binned scatter: smoke days vs PM2.5 residuals
panel_iv["pm25_resid"] = panel_iv["pm25_annual_mean"] - panel_iv.groupby("leaid")["pm25_annual_mean"].transform("mean")
panel_iv["smoke_resid"] = panel_iv["smoke_days"] - panel_iv.groupby("leaid")["smoke_days"].transform("mean")

bins = pd.qcut(panel_iv["smoke_resid"], 20, duplicates="drop")
binned = panel_iv.groupby(bins, observed=True)[["smoke_resid","pm25_resid"]].mean().reset_index()

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(binned["smoke_resid"], binned["pm25_resid"], s=60, c="steelblue")
ax.set_xlabel("Smoke Days (demeaned)")
ax.set_ylabel("PM2.5 (demeaned)")
ax.set_title("First Stage: Smoke Days -> PM2.5\n(within-district variation; binned scatter)")
plt.tight_layout()
plt.savefig(OUT_DIR / "04_first_stage.png", bbox_inches="tight")
plt.show()